# DeepTrace - Training Notebook
**Owner: Zihao Li**

Trains the MLP head on top of frozen CLIP ViT-B/32 features.

Three training modes:
- `faceswap` - train on Celeb-DF v2 only
- `ai_video` - train on DeepTrace-GV only
- `joint`    - train on both (primary experiment)

**Runtime: GPU (T4)**

## Step 1 - Install dependencies

In [1]:
!pip install open-clip-torch ftfy regex scikit-learn tqdm wandb gdown -q
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
Done


## Step 2 - Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 3 - Copy model files over



In [3]:
import os
import shutil

dst = '/content/deeptrace/src/model/'

# Skip if files already present
files = ['clip_extractor.py', 'mlp_head.py', 'train.py']
already_present = all(os.path.exists(f'{dst}{f}') for f in files)

if already_present:
    print("Model files already present, skipping.")
else:
    src = '/content/drive/MyDrive/DeepTrace-GV/model_code/'
    os.makedirs(dst, exist_ok=True)
    for f in files:
        shutil.copy(f'{src}{f}', f'{dst}{f}')
        print(f'Copied {f}')

print('Model files:')
for f in os.listdir(dst):
    print(f'  {f}')

Copied clip_extractor.py
Copied mlp_head.py
Copied train.py
Model files:
  train.py
  mlp_head.py
  clip_extractor.py


## Step 4 - Download Celeb-DF v2

In [4]:
import os
import shutil
import zipfile

zip_dst = '/content/drive/MyDrive/DeepTrace-GV/Kaggle/CelebsDF-v2/Celeb-DF-v2.zip'
extract_dst = '/content/celeb-df-v2/'

# Skip if already extracted
if os.path.exists(extract_dst) and os.listdir(extract_dst):
    print("Celeb-DF v2 already extracted, skipping.")
else:
    # Skip copy if zip already present
    if not os.path.exists(zip_dst):
        print('Copying from Drive...')
        shutil.copy(
            '/content/drive/MyDrive/Kaggle/CelebsDF-v2/Celeb-DF-v2.zip',
            zip_dst
        )
        print('Done copying.')
    else:
        print('Zip already present, skipping copy.')

    print('Unzipping...')
    with zipfile.ZipFile(zip_dst, 'r') as z:
        z.extractall(extract_dst)
    os.remove(zip_dst)
    print('Done.')

print('Contents:')
for item in os.listdir(extract_dst):
    print(f'  {item}')

Copying from Drive...
Done copying.
Unzipping...
Done.
Contents:
  YouTube-real
  List_of_testing_videos.txt
  Celeb-synthesis
  Celeb-real


## Step 5 - Verify GPU and imports

In [5]:
import torch
import open_clip
import sys

sys.path.insert(0, '/content/deeptrace/src/model')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

from clip_extractor import CLIPExtractor
from mlp_head import MLPHead, DeepTrace

device = 'cuda' if torch.cuda.is_available() else 'cpu'
extractor = CLIPExtractor().to(device)
print(f'CLIP loaded. Feature dim: 512')

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP loaded. Feature dim: 512


## Step 6 - Verify dataset

In [6]:
import pandas as pd

MANIFEST = '/content/drive/MyDrive/DeepTrace-GV/staging_manifest.csv'
DEEPTRACE_ROOT = '/content/drive/MyDrive/DeepTrace-GV/data/reencoded'
CELEBDF_ROOT = '/content/celeb-df-v2'

manifest = pd.read_csv(MANIFEST)
print('Manifest loaded:')
print(manifest.groupby(['generator', 'split']).size())
print(f'\nTotal: {len(manifest)} clips')

# Verify Celeb-DF
list_path = f'{CELEBDF_ROOT}/List_of_testing_videos.txt'
with open(list_path) as f:
    lines = f.readlines()
print(f'\nCeleb-DF v2 test list: {len(lines)} clips')

Manifest loaded:
generator  split
kling      test      7
           train    29
           val       6
real       test     19
           train    91
           val      20
sora       test      5
           train    23
           val       5
veo        test      6
           train    29
           val       6
dtype: int64

Total: 246 clips

Celeb-DF v2 test list: 518 clips


## Step 7 - Train all three variants

Run each cell in sequence. Each training run takes ~10-15 minutes on T4.
Checkpoints are saved to Drive automatically.

In [7]:
# Add src to path
import sys
sys.path.insert(0, '/content/deeptrace/src/model')
sys.path.insert(0, '/content/deeptrace/src')

CHECKPOINT_DIR = '/content/drive/MyDrive/DeepTrace-GV/checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# --- Variant 1: AI-video only ---
!python /content/deeptrace/src/model/train.py \
    --mode ai_video \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs 20 \
    --no-wandb

object address  : 0x7867fb5b4940
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [ ]:
MANIFEST = '/content/drive/MyDrive/DeepTrace-GV/staging_manifest.csv'
DEEPTRACE_ROOT = '/content/drive/MyDrive/DeepTrace-GV/data/reencoded'
CELEBDF_ROOT = '/content/celeb-df-v2'
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepTrace-GV/checkpoints'

# Variant 2 - Face-swap only
!python /content/deeptrace/src/model/train.py \
    --mode faceswap \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs 20 \
    --no-wandb

# Variant 3 - Joint
!python /content/deeptrace/src/model/train.py \
    --mode joint \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs 20 \
    --no-wandb

Device: cuda

Dataset (faceswap mode):
  Train: 414 clips (real=140, fake=274)
  Val:   104 clips
  Test:  0 clips
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Trainable parameters: 66,178

Training (faceswap mode, 20 epochs)...

Epoch  1/20 | Train Loss: 0.7312 AUC: 0.5154 | Val Loss: 0.6767 AUC: 0.5754 | Time: 6.9s
  Saved best model (val AUC=0.5754)
Epoch  2/20 | Train Loss: 0.6688 AUC: 0.6189 | Val Loss: 0.6711 AUC: 0.6898 | Time: 6.5s
  Saved best model (val AUC=0.6898)
Epoch  3/20 | Train Loss: 0.6176 AUC: 0.7073 | Val Loss: 0.6576 AUC: 0.7687 | Time: 6.7s
  Saved best model (val AUC=0.7687)
Epoch  4/20 | Train Loss: 0.5686 AUC: 0.7712 | Val Loss: 0.6324 AUC: 0.8018 | Time: 6.5s
  Saved best model (val AUC=0.8018)
Epoch  5/20 | Train Loss: 0.5606 AUC: 0.7708 | Val Loss: 0.5903 AUC: 0.8090 | Time: 6.6s
  Saved best model

## Step 8 - Verify checkpoints saved

In [8]:
import os
checkpoints = os.listdir(CHECKPOINT_DIR)
print('Saved checkpoints:')
for f in sorted(checkpoints):
    size = os.path.getsize(f'{CHECKPOINT_DIR}/{f}') / 1024
    print(f'  {f} ({size:.1f} KB)')

Saved checkpoints:
  deeptrace_ai_video_full_best.pth (264.6 KB)
  deeptrace_faceswap_full_best.pth (264.6 KB)
  deeptrace_joint_full_best.pth (264.5 KB)
  deeptrace_joint_kling_best.pth (264.5 KB)
  deeptrace_joint_sora_best.pth (264.5 KB)
  deeptrace_joint_veo_best.pth (264.5 KB)


## Step 9 - Build the ONNX Export pipeline

In [9]:
# Step 9 - ONNX Export + Speed Benchmark

!pip install onnxscript onnxruntime -q

import torch
import numpy as np
import time
import sys
import os

sys.path.insert(0, '/content/deeptrace/src/model')
from clip_extractor import CLIPExtractor
from mlp_head import MLPHead, DeepTrace

device = 'cuda'
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepTrace-GV/checkpoints'
ONNX_DIR = '/content/drive/MyDrive/DeepTrace-GV/onnx'
os.makedirs(ONNX_DIR, exist_ok=True)

# --- Export all three variants ---
variants = ['ai_video', 'faceswap', 'joint']

for variant in variants:
    ckpt_path = f'{CHECKPOINT_DIR}/deeptrace_{variant}_full_best.pth'
    onnx_path = f'{ONNX_DIR}/deeptrace_{variant}_head.onnx'

    if os.path.exists(onnx_path):
        print(f'[skip] {variant} already exported')
        continue
    if not os.path.exists(ckpt_path):
        print(f'[skip] {variant} checkpoint not found')
        continue

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    head = MLPHead().to(device)
    head.load_state_dict(ckpt['head_state'])
    head.eval()

    dummy_input = torch.randn(1, 512).to(device)
    torch.onnx.export(
        head,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=18,
        input_names=['clip_features'],
        output_names=['logits'],
        dynamic_axes={
            'clip_features': {0: 'batch_size'},
            'logits': {0: 'batch_size'}
        }
    )
    print(f'[ok] Exported {variant} -> {onnx_path}')

print('\nAll exports done.')
print('ONNX files:')
for f in os.listdir(ONNX_DIR):
    size = os.path.getsize(f'{ONNX_DIR}/{f}') / 1024
    print(f'  {f} ({size:.1f} KB)')

# --- Speed Benchmark (joint variant) ---
print('\n--- Speed Benchmark (joint variant, T4 GPU) ---')

import onnxruntime as ort

sess = ort.InferenceSession(
    f'{ONNX_DIR}/deeptrace_joint_head.onnx',
    providers=['CPUExecutionProvider']
)

extractor = CLIPExtractor().to(device)
extractor.eval()

n_clips = 100
frames_per_clip = 5
times = []
dummy_frame = torch.randn(frames_per_clip, 3, 224, 224).to(device)

# Warmup
for _ in range(5):
    with torch.no_grad():
        features = extractor(dummy_frame)
    features_np = features.cpu().numpy()
    sess.run(None, {'clip_features': features_np})

# Benchmark
for _ in range(n_clips):
    t0 = time.time()
    with torch.no_grad():
        features = extractor(dummy_frame)
    features_np = features.cpu().numpy()
    logits = sess.run(None, {'clip_features': features_np})[0]
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    clip_prob = probs[:, 1].mean()
    times.append(time.time() - t0)

mean_ms = np.mean(times) * 1000
std_ms  = np.std(times) * 1000
fps     = frames_per_clip / np.mean(times)

print(f'Clips tested:        {n_clips}')
print(f'Frames per clip:     {frames_per_clip}')
print(f'Mean latency:        {mean_ms:.1f} ms per clip')
print(f'Std:                 {std_ms:.1f} ms')
print(f'Throughput:          {fps:.1f} FPS')
print(f'Sub-second target:   {"PASS ✓" if mean_ms < 1000 else "FAIL ✗"}')

# Save benchmark results
import json

results = {
    "model": "DeepTrace joint variant (CLIP ViT-B/32 + MLP head)",
    "hardware": "T4 GPU (CLIP) + CPU (ONNX MLP)",
    "frames_per_clip": frames_per_clip,
    "n_clips_tested": n_clips,
    "mean_latency_ms": round(mean_ms, 2),
    "std_latency_ms": round(std_ms, 2),
    "throughput_fps": round(fps, 2),
    "sub_second_pass": bool(mean_ms < 1000)
}

results_path = '/content/drive/MyDrive/DeepTrace-GV/onnx/speed_benchmark.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {results_path}')
print(json.dumps(results, indent=2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.6 MB/s eta 0:00:00
[skip] ai_video already exported
[skip] faceswap already exported
[skip] joint already exported

All exports done.
ONNX files:
  deeptrace_ai_video_head.onnx.data (257.5 KB)
  deeptrace_ai_video_head.onnx (5.4 KB)
  deeptrace_faceswap_head.onnx.data (257.5 KB)
  deeptrace_faceswap_head.onnx (5.4 KB)
  deeptrace_joint_head.onnx.data (257.5 KB)
  deeptrace_joint_head.onnx (5.4 KB)
  speed_benchmark.json (0.3 KB)

--- Speed Benchmark (joint variant, T4 GPU) ---


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Clips tested:        100
Frames per clip:     5
Mean latency:        14.2 ms per clip
Std:                 2.3 ms
Throughput:          352.8 FPS
Sub-second target:   PASS ✓
Results saved to /content/drive/MyDrive/DeepTrace-GV/onnx/speed_benchmark.json
{
  "model": "DeepTrace joint variant (CLIP ViT-B/32 + MLP head)",
  "hardware": "T4 GPU (CLIP) + CPU (ONNX MLP)",
  "frames_per_clip": 5,
  "n_clips_tested": 100,
  "mean_latency_ms": 14.17,
  "std_latency_ms": 2.26,
  "throughput_fps": 352.8,
  "sub_second_pass": true
}


In [10]:
import shutil
shutil.copy(
    '/content/drive/MyDrive/DeepTrace-GV/onnx/speed_benchmark.json',
    '/content/speed_benchmark.json'
)
print('Done')

Done


## Step 10 - LOGO Evaluation Rounds (all three, hardcoded paths)

In [ ]:
## Step 10 - LOGO Evaluation Rounds (all three, hardcoded paths)

MANIFEST       = '/content/drive/MyDrive/DeepTrace-GV/staging_manifest.csv'
DEEPTRACE_ROOT = '/content/drive/MyDrive/DeepTrace-GV/data/reencoded'
CELEBDF_ROOT   = '/content/celeb-df-v2'
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepTrace-GV/checkpoints'

# Round 1: train on Kling + Veo, test on Sora
!python /content/deeptrace/src/model/train.py \
    --mode joint \
    --logo-held-out sora \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir /content/checkpoints_local \
    --epochs 20 \
    --no-wandb

# Round 2: train on Sora + Veo, test on Kling
!python /content/deeptrace/src/model/train.py \
    --mode joint \
    --logo-held-out kling \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir /content/checkpoints_local \
    --epochs 20 \
    --no-wandb

# Round 3: train on Sora + Kling, test on Veo
!python /content/deeptrace/src/model/train.py \
    --mode joint \
    --logo-held-out veo \
    --manifest {MANIFEST} \
    --deeptrace-root {DEEPTRACE_ROOT} \
    --celebdf-root {CELEBDF_ROOT} \
    --checkpoint-dir /content/checkpoints_local \
    --epochs 20 \
    --no-wandb

# Copy all three from local to Drive
import shutil, os
for gen in ['sora', 'kling', 'veo']:
    src = f'/content/checkpoints_local/deeptrace_joint_{gen}_best.pth'
    dst = f'/content/drive/MyDrive/DeepTrace-GV/checkpoints/deeptrace_joint_{gen}_best.pth'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {gen} checkpoint to Drive')

Device: cuda

Dataset (joint mode):
  Train: 563 clips (real=231, fake=332)
  Val:   141 clips
  Test:  37 clips
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Trainable parameters: 66,178

Training (joint mode, 20 epochs)...

Epoch  1/20 | Train Loss: 0.6728 AUC: 0.6152 | Val Loss: 0.6792 AUC: 0.7283 | Time: 24.9s
  Saved best model (val AUC=0.7283)
Epoch  2/20 | Train Loss: 0.5862 AUC: 0.7458 | Val Loss: 0.6613 AUC: 0.7788 | Time: 24.1s
  Saved best model (val AUC=0.7788)
Epoch  3/20 | Train Loss: 0.5498 AUC: 0.7796 | Val Loss: 0.6174 AUC: 0.7867 | Time: 24.8s
  Saved best model (val AUC=0.7867)
Epoch  4/20 | Train Loss: 0.5275 AUC: 0.8043 | Val Loss: 0.5412 AUC: 0.8110 | Time: 25.0s
  Saved best model (val AUC=0.8110)
Epoch  5/20 | Train Loss: 0.5038 AUC: 0.8247 | Val Loss: 0.4831 AUC: 0.8442 | Time: 25.2s
  Saved best model

In [ ]:
import shutil
shutil.copy(
    '/content/checkpoints_local/deeptrace_joint_sora_best.pth',
    '/content/drive/MyDrive/DeepTrace-GV/checkpoints/deeptrace_joint_sora_best.pth'
)
print("Copied")

Copied


In [ ]:
import torch
ckpt = torch.load('/content/drive/MyDrive/DeepTrace-GV/checkpoints/deeptrace_joint_sora_best.pth', map_location='cpu', weights_only=False)
print(f"val_auc={ckpt['val_auc']:.4f}, epoch={ckpt['epoch']}")

val_auc=0.9941, epoch=8


In [16]:
# Step 10b - Correct held-out test AUC
import torch, csv, sys, os
import numpy as np
from torch.utils.data import DataLoader

sys.path.insert(0, '/content/deeptrace/src/model')
from clip_extractor import CLIPExtractor
from mlp_head import MLPHead, DeepTrace
from train import VideoClipDataset, CLIP_TRANSFORM, evaluate

device = 'cuda'
MANIFEST = '/content/drive/MyDrive/DeepTrace-GV/staging_manifest.csv'
DEEPTRACE_ROOT = '/content/drive/MyDrive/DeepTrace-GV/data/reencoded'
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepTrace-GV/checkpoints'

extractor = CLIPExtractor().to(device)
extractor.eval()

# Load real test clips (shared across all LOGO rounds)
real_test = []
with open(MANIFEST, newline='') as f:
    for row in csv.DictReader(f):
        if row['split'] != 'test' or row['generator'] != 'real':
            continue
        path = os.path.join(DEEPTRACE_ROOT, 'real', f"{row['clip_id']}.mp4")
        if os.path.exists(path):
            real_test.append((path, 0))  # 0 = real

print(f"Real test clips: {len(real_test)}")

logo_results = {}
for generator in ['sora', 'kling', 'veo']:
    # Held-out generator fake clips + real clips
    fake_test = []
    with open(MANIFEST, newline='') as f:
        for row in csv.DictReader(f):
            if row['split'] != 'test' or row['generator'] != generator:
                continue
            path = os.path.join(DEEPTRACE_ROOT, generator, f"{row['clip_id']}.mp4")
            if os.path.exists(path):
                fake_test.append((path, 1))  # 1 = fake

    samples = fake_test + real_test
    print(f"\n{generator}: {len(fake_test)} fake + {len(real_test)} real = {len(samples)} total")

    ckpt = torch.load(f'{CHECKPOINT_DIR}/deeptrace_joint_{generator}_best.pth',
                      map_location=device, weights_only=False)
    head = MLPHead().to(device)
    head.load_state_dict(ckpt['head_state'])
    model = DeepTrace(extractor, head).to(device)

    loader = DataLoader(VideoClipDataset(samples, CLIP_TRANSFORM, augment=False),
                        batch_size=8, shuffle=False)
    metrics = evaluate(model, loader, device)
    logo_results[generator] = metrics
    print(f"  AUC={metrics['auc']:.4f}, Acc={metrics['acc']:.4f}")

aucs = [v['auc'] for v in logo_results.values()]
print(f"\nLOGO mean AUC (held-out test + real): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print("Caveat: 5-7 fake clips + 19 real clips per round - exploratory only.")

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Real test clips: 19

sora: 5 fake + 19 real = 24 total
  AUC=1.0000, Acc=1.0000

kling: 7 fake + 19 real = 26 total
  AUC=0.9850, Acc=0.8846

veo: 6 fake + 19 real = 25 total
  AUC=0.9386, Acc=0.8800

LOGO mean AUC (held-out test + real): 0.9745 ± 0.0261
Caveat: 5-7 fake clips + 19 real clips per round - exploratory only.
